# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadhany222/flyrank-ml-assignment1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why


My lane (Refresh/Content Opportunity Scoring) is a **ranking** question — "which pages first,"
not a plain yes/no. Per the toolkit table, ranking questions need a classifier's probability
evaluated at precision@K, not just a label.

I'm starting with a **Decision Tree** (max_depth=3), because I already have direct evidence
from notebook 02 that a small, readable tree beats a hand rule on this exact data — Precision@50
went from 0.24 (rule) to 0.72 (depth-3 tree) in that experiment. A depth-3 tree is also something
I can print and read end-to-end, which matters more than squeezing out a couple more points with
a black-box model. I'll also fit a **Random Forest** as a stronger second option, since the
reference pipeline shows it doing meaningfully better (Precision@50 = 0.740) — and report both,
since the skill's rule is "simplicity is a feature; add complexity only when the comparison earns it."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Client-grouped split** — `GroupShuffleSplit` on `client_id`, not a plain random split. Pages
from the same client can share patterns (site-wide template, niche, general health) that a
model could memorize rather than generalize from. If pages from the same client appear in both
train and test, the model could get an easy, dishonest boost just from recognizing "this is
Client X's writing style," not from learning real decline signals. This matches the lane guide's
explicit validation rule: "client/group holdout, when pages from the same client may share
patterns the model could memorize."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

Same data (starter CSV), same label (`trend_direction == "down"`), same client-grouped test
split. Important finding before trusting any comparison number: **zero rows in the test split
have a non-zero baseline score** (my w04 rule only ever flagged 26 pages total out of 30,000,
and by chance none of them landed in this split's 7 test clients). This means Precision@50 for
the baseline on this split is not measuring the rule — it's measuring tie-break order among
5,078 identically-scored rows. That number (0.600) is noise, not baseline skill, and I'm
reporting it as such rather than treating it as a real result.

**Does complexity earn its keep? Inconclusive on Precision@50 — and that's the real finding.**
My baseline rule is too narrow to fairly compare at K=50: it only ever flags 26 pages across the
entire 30,000-row dataset, and none of them fell in this test split's 7 clients, making its
"Precision@50" here pure tie-break noise (0 non-zero scores in the test set). The decision tree
(0.500) and random forest (0.520) both did modestly better than the pure base rate would suggest
is trivial, but neither dramatically beat the ~0.52 base rate either — meaning on THIS
client-holdout split, with THESE features, extra model complexity bought very little. The
honest conclusion isn't "the model wins" or "the baseline wins" — it's that my Week-4 baseline
rule is too narrow (3 strict conditions ANDed together) to serve as a meaningful comparison
point at K=50, and a fairer baseline test would need either a much larger test split, a looser
rule that flags more candidates, or evaluating Precision@K at a K matched to how many pages the
rule actually flags (K≈26, not K=50).

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadhany222/flyrank-ml-assignment1"
REPO_DIR = "flyrank-ml-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 100).astype(int)
declining_flag = (df["trend_direction"] == "down").astype(int)
df["baseline_score"] = stale * visible_flag * declining_flag * df["impressions_90d"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining"].mean()
baseline_p50 = precision_at_k(df["baseline_score"].values, df["is_declining"].values, 50)
print(f"Base rate (always predict majority): {base_rate:.3f}")
print(f"Baseline Precision@50: {baseline_p50:.3f}")

Base rate (always predict majority): 0.542
Baseline Precision@50: 0.740


In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

feature_cols = ["impressions_90d", "days_since_last_update", "avg_position", "ctr",
                 "content_age_days", "word_count", "sessions_90d", "engagement_rate"]

model_df = df.dropna(subset=feature_cols + ["is_declining", "client_id"]).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["client_id"]))

X_train, X_test = model_df.iloc[train_idx][feature_cols], model_df.iloc[test_idx][feature_cols]
y_train, y_test = model_df.iloc[train_idx]["is_declining"], model_df.iloc[test_idx]["is_declining"]

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Unique clients train: {model_df.iloc[train_idx]['client_id'].nunique()}, "
      f"test: {model_df.iloc[test_idx]['client_id'].nunique()}")

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced",
                              random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = model_df.iloc[test_idx]["baseline_score"].values
y_test_arr = y_test.values

baseline_test_p50 = precision_at_k(baseline_test_scores, y_test_arr, 50)
tree_p50 = precision_at_k(tree_scores, y_test_arr, 50)
rf_p50 = precision_at_k(rf_scores, y_test_arr, 50)
test_base_rate = y_test_arr.mean()

comparison = pd.DataFrame({
    "method": ["base rate (majority)", "baseline rule", "decision tree (depth=3)", "random forest"],
    "precision_at_50": [test_base_rate, baseline_test_p50, tree_p50, rf_p50]
}).round(3)

print("\n=== Model vs Baseline comparison (same client-holdout test split) ===")
print(comparison.to_string(index=False))

Train rows: 17223, Test rows: 5078
Unique clients train: 25, test: 7

=== Model vs Baseline comparison (same client-holdout test split) ===
                 method  precision_at_50
   base rate (majority)            0.521
          baseline rule            0.600
decision tree (depth=3)            0.500
          random forest            0.520


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [3]:
from sklearn.tree import export_text

print("--- Readable depth-3 tree ---")
print(export_text(tree, feature_names=feature_cols))

# Feature importance from the random forest
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n--- Random Forest feature importance ---")
print(importances.round(3))

# Look at 3 concrete wrong cases from the RF's top-50
test_df = model_df.iloc[test_idx].copy()
test_df["rf_score"] = rf_scores
test_df_sorted = test_df.sort_values("rf_score", ascending=False)

wrong_in_top50 = test_df_sorted.head(50)[test_df_sorted.head(50)["is_declining"] == 0]
print(f"\nWrong predictions in RF's top 50: {len(wrong_in_top50)}")
wrong_in_top50[["content_id", "rf_score", "impressions_90d", "days_since_last_update",
                 "avg_position", "ctr", "is_declining"]].head(3)

--- Readable depth-3 tree ---
|--- impressions_90d <= 7.50
|   |--- avg_position <= 1.65
|   |   |--- days_since_last_update <= 221.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  221.00
|   |   |   |--- class: 1
|   |--- avg_position >  1.65
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  7.50
|   |--- ctr <= 0.30
|   |   |--- content_age_days <= 342.50
|   |   |   |--- class: 1
|   |   |--- content_age_days >  342.50
|   |   |   |--- class: 0
|   |--- ctr >  0.30
|   |   |--- sessions_90d <= 69.50
|   |   |   |--- class: 1
|   |   |--- sessions_90d >  69.50
|   |   |   |--- class: 0


--- Random Forest feature importance ---
impressions_90d           0.369
avg_position              0.218
content_age_days          0.109
word_count                0.093
days_since_last_update    0.069
ctr                       0.068
sessions_90d              0.054
engagement_ra

,content_id,rf_score,impressions_90d,days_since_last_update,avg_position,ctr,is_declining
2357,content_8f1409b2674e,0.756862,209,104,20.0,0.0,0
20736,content_41baf0722ad9,0.753357,3115,104,12.8,0.0,0
4050,content_500bd3907331,0.752502,4037,104,5.5,0.1,0


In [4]:
n_nonzero_baseline_test = (baseline_test_scores > 0).sum()
print(f"Test rows with baseline_score > 0: {n_nonzero_baseline_test} out of {len(baseline_test_scores)}")



Test rows with baseline_score > 0: 0 out of 5078


In [5]:

full_baseline_p50 = precision_at_k(df["baseline_score"].values, df["is_declining"].values, 50)
print(f"Baseline Precision@50 on FULL dataset (not train/test split): {full_baseline_p50:.3f}")
print(f"(Recall: baseline only flags {(df['baseline_score']>0).sum()} pages total, so K=50 still exceeds its real candidate pool)")

Baseline Precision@50 on FULL dataset (not train/test split): 0.740
(Recall: baseline only flags 26 pages total, so K=50 still exceeds its real candidate pool)


In [6]:
K_fair = (df["baseline_score"] > 0).sum()
baseline_p_fair = precision_at_k(df["baseline_score"].values, df["is_declining"].values, K_fair)
tree_p_fair = precision_at_k(tree.predict_proba(model_df[feature_cols])[:, 1], model_df["is_declining"].values, K_fair)
rf_p_fair = precision_at_k(rf.predict_proba(model_df[feature_cols])[:, 1], model_df["is_declining"].values, K_fair)

print(f"Fair comparison at K={K_fair} (matched to baseline's real flag count):")
print(f"  Baseline Precision@{K_fair}: {baseline_p_fair:.3f}")
print(f"  Tree Precision@{K_fair}:     {tree_p_fair:.3f}")
print(f"  RF Precision@{K_fair}:       {rf_p_fair:.3f}")

Fair comparison at K=26 (matched to baseline's real flag count):
  Baseline Precision@26: 1.000
  Tree Precision@26:     0.731
  RF Precision@26:       0.962


**What the model leans on:** [fill in once you see the printed importances — name the top 2-3
features and whether they make intuitive sense, e.g. "avg_position and ctr dominate, which
matches Signal 2's CONFIRMED verdict from w04"]

**Where it's wrong:** [fill in using the 3 printed wrong cases — describe what these
false-positive pages have in common, e.g. high impressions but not actually declining, or
borderline threshold cases]

**Does complexity earn its keep?** [compare tree_p50 vs rf_p50 from the table above — if RF
only marginally beats the depth-3 tree, say so plainly; don't reward complexity just because
it's fancier]

**Confirming the full picture:** on the full 30,000-row dataset, baseline Precision@50 = 0.740
(37/50 correct) — but the rule only ever confidently flags **26 pages**. That means 26 of those
37 hits are real, rule-driven flags (and by construction, every flagged page IS declining), while
the remaining 11 are tie-break luck among thousands of identically-zero-scored rows. Precision@50
simply isn't a fair metric for a rule this narrow — it silently rewards zero-score tie-breaking
past the point where the rule has any real opinion. The honest comparison would be
**Precision@26** (matched to the rule's real candidate pool), not Precision@50. This is the
central finding of this notebook: before declaring a model "better" or "worse" than a baseline,
the metric's K has to match what the baseline can actually produce — otherwise the comparison
measures noise, not skill.


**The fair, matched comparison (K=26):**

| Method | Precision@26 |
|---|---|
| Baseline rule | 1.000 |
| Decision tree (depth=3) | 0.731 |
| Random forest | 0.962 |

The baseline hits 1.000 by construction — every page it flags IS declining, since the rule's
scoring formula (`stale × visible × declining × impressions`) multiplies by the declining flag
itself. This isn't really "predictive skill," it's closer to a filter that only ever returns
positives — which is honest, but also means a perfect score here doesn't mean much beyond "the
rule is internally consistent." Random forest comes close (0.962), which is a genuinely
interesting result: without hard-coding `trend_direction` into the score at all, RF's top 26
predictions are almost as clean as the baseline's guaranteed-correct list — meaning the model
is finding real signal, not noise.

**One important caveat:** this fair comparison was computed on the full dataset (train + test
combined) for the tree and random forest, not the held-out client-split test set used in
Section 3 — so these two numbers are partly in-sample and likely optimistic compared to true
generalization performance. The honest headline result of this notebook isn't a single winning
number; it's that **K must match what a method can actually produce** before any comparison is
meaningful, and that a narrow rule scoring "perfect" is a different kind of result than a model
approaching that same precision without being handed the label directly.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.